# 3_projects/timezone: The client

**`client.py` is the host: The app that connects the server to an OpenAI model. It hands each MCP primitive to the one who owns it. The resource goes into the system message, the prompt becomes the user's message, and the tool goes to the model, which decides when to call it.**

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 server.py              client.py (the host)                 OpenAI model
 ---------              --------------------                 ------------
 zones://offices  ---&gt;  system message  (the app chose)   ---&gt;
 plan_meeting     ---&gt;  user message    (the user chose)  ---&gt;  reads both
 convert_time     ---&gt;  tools=[...]     (the model chooses) &lt;--&gt; tool_calls
</pre>

Read `Server_Implementation_Details.ipynb` and `../../1_crash-course/3-openai-integration/client-functions-one-question.ipynb` first. The tool bridge here is the same as there, plus a loop.

The server must be running (`python server.py` in this folder), and `.env` in this folder must hold `OPENAI_API_KEY`.

## 1. Setup

In [1]:
import json

from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from openai import AsyncOpenAI

load_dotenv()

openai_client = AsyncOpenAI()
model = "gpt-4o-mini"
url = "http://127.0.0.1:8060/mcp"

- **`load_dotenv()`:** Puts `OPENAI_API_KEY` from `.env` into the environment, where `AsyncOpenAI()` looks for it.
- **`AsyncOpenAI`:** The async OpenAI client, so its calls can be awaited inside the MCP session.

## 2. The tool, in OpenAI's format

OpenAI's API doesn't know MCP. It takes tools in its own format, `{"type": "function", "function": {...}}`. `get_mcp_tools` asks the server for its tools and rewrites each one. The MCP `inputSchema` is already JSON Schema, so it goes into `parameters` unchanged:

In [2]:
async def get_mcp_tools(session):
    tools_result = await session.list_tools()

    tools = []
    for tool in tools_result.tools:
        tools.append(
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema,
                },
            }
        )

    return tools

## 3. The tool loop

**The model can ask for tools several times before it answers. `run_conversation` keeps going until a reply has no tool calls.**

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 ask the model  ---&gt;  tool_calls?  --no--&gt;  return its text (the answer)
      ^                    | yes
      |                    v
      +----  add each result  &lt;---  session.call_tool(name, args)
</pre>

In [3]:
async def run_conversation(session, messages, tools):
    while True:
        response = await openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
        )
        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = await session.call_tool(name, args)
            print("The model called:", name, args, "->", result.content[0].text)
            messages.append(
                {"role": "tool", "tool_call_id": tool_call.id, "content": result.content[0].text}
            )

- **`messages.append(message)`:** The model's reply goes into the history, tool calls included, so the next request shows the model what it asked for.
- **`json.loads(tool_call.function.arguments)`:** The model sends the arguments as a JSON string. `json.loads` turns it into a dict for `call_tool`.
- **`"tool_call_id": tool_call.id`:** Each result names the call it answers, so the model can match results to calls when it asks for several at once.
- **Part 3's `client-functions-one-question.py` allowed one round of tool calls,** then forced a text answer with `tool_choice="none"`. This loop allows any number of rounds. A production host would also cap them, so a confused model can't loop forever.

## 4. One question, using all three primitives

The resource and the prompt are fetched first, then the conversation runs:

In [4]:
async with streamable_http_client(url) as (read, write, get_session_id):
    async with ClientSession(read, write) as session:
        await session.initialize()

        offices = await session.read_resource("zones://offices")
        tools = await get_mcp_tools(session)
        prompt = await session.get_prompt("plan_meeting", {"date": "2026-10-28", "time": "15:00"})

        messages = [
            {"role": "system", "content": "Our offices and their time zones:\n" + offices.contents[0].text},
            {"role": "user", "content": prompt.messages[0].content.text},
        ]
        answer = await run_conversation(session, messages, tools)

The model called: convert_time {'time': '2026-10-28 15:00', 'from_zone': 'Asia/Kolkata', 'to_zone': 'Europe/London'} -> 2026-10-28 09:30 GMT
The model called: convert_time {'time': '2026-10-28 15:00', 'from_zone': 'Asia/Kolkata', 'to_zone': 'America/New_York'} -> 2026-10-28 05:30 EDT


The model called `convert_time` twice in one reply, once per office. It took the zone names from the system message, since the prompt only said "every office".

The three pieces it was given. First the system message, built from the resource. The app chose to put it there:

In [5]:
print(messages[0]["content"])

Our offices and their time zones:
Pune: Asia/Kolkata
London: Europe/London
New York: America/New_York


The user's message, which is the prompt's text:

In [6]:
print(messages[1]["content"])

We want a meeting on 2026-10-28 at 15:00 Pune time. Use the convert_time tool to find its local time in every office, and say whether it falls inside working hours, 09:00 to 18:00 local time.


And the tool, after `get_mcp_tools` rewrote it:

In [7]:
print(json.dumps(tools[0], indent=2))

{
  "type": "function",
  "function": {
    "name": "convert_time",
    "description": "Convert a time from one time zone to another. time is \"YYYY-MM-DD HH:MM\", and the zones are IANA names like \"Asia/Kolkata\".",
    "parameters": {
      "properties": {
        "time": {
          "title": "Time",
          "type": "string"
        },
        "from_zone": {
          "title": "From Zone",
          "type": "string"
        },
        "to_zone": {
          "title": "To Zone",
          "type": "string"
        }
      },
      "required": [
        "time",
        "from_zone",
        "to_zone"
      ],
      "title": "convert_timeArguments",
      "type": "object"
    }
  }
}


## 5. What the conversation looked like

`messages` started with 2 entries, and the loop added the rest:

In [8]:
print(len(messages))
print(messages[2].model_dump_json(indent=2, exclude_none=True))

6
{
  "role": "assistant",
  "annotations": [],
  "tool_calls": [
    {
      "id": "call_V4v4HLyFooESO01N2jnadXfS",
      "function": {
        "arguments": "{\"time\": \"2026-10-28 15:00\", \"from_zone\": \"Asia/Kolkata\", \"to_zone\": \"Europe/London\"}",
        "name": "convert_time"
      },
      "type": "function"
    },
    {
      "id": "call_VsFulEwEFpO4uLXXXpUZXhnf",
      "function": {
        "arguments": "{\"time\": \"2026-10-28 15:00\", \"from_zone\": \"Asia/Kolkata\", \"to_zone\": \"America/New_York\"}",
        "name": "convert_time"
      },
      "type": "function"
    }
  ]
}


- **`messages[2]`:** The model's first reply. It has no text, so `content` is `None` and `exclude_none` leaves it out. `tool_calls` holds two calls. Their `arguments` are JSON strings, which is why the loop uses `json.loads`.
- **`messages[3]` and `messages[4]`:** The two tool results, one per call, each with its `tool_call_id`.
- **`messages[5]`:** The model's second reply, with no tool calls, so the loop returned it.

The answer:

In [9]:
print(answer)

The meeting on 2026-10-28 at 15:00 Pune time converts to:

- **London time**: 2026-10-28 at 09:30 GMT
- **New York time**: 2026-10-28 at 05:30 EDT

Now, let's check if these times fall within the working hours of 09:00 to 18:00 local time for each office:

- **London**: 09:30 is within working hours (09:00 to 18:00).
- **New York**: 05:30 is outside working hours (before 09:00).

In summary:
- London: Within working hours
- New York: Outside working hours


## 6. A week later, after the US clock change

The same prompt for 5 November. Only the date changes:

In [10]:
async with streamable_http_client(url) as (read, write, get_session_id):
    async with ClientSession(read, write) as session:
        await session.initialize()

        prompt = await session.get_prompt("plan_meeting", {"date": "2026-11-05", "time": "15:00"})
        messages = [
            {"role": "system", "content": "Our offices and their time zones:\n" + offices.contents[0].text},
            {"role": "user", "content": prompt.messages[0].content.text},
        ]
        answer = await run_conversation(session, messages, tools)

print(answer)

The model called: convert_time {'time': '2026-11-05 15:00', 'from_zone': 'Asia/Kolkata', 'to_zone': 'Europe/London'} -> 2026-11-05 09:30 GMT
The model called: convert_time {'time': '2026-11-05 15:00', 'from_zone': 'Asia/Kolkata', 'to_zone': 'America/New_York'} -> 2026-11-05 04:30 EST
The meeting scheduled for **2026-11-05 at 15:00 Pune time** converts to:

- **London Time:** 2026-11-05 at 09:30 GMT (inside working hours: 09:00 - 18:00)
- **New York Time:** 2026-11-05 at 04:30 EST (outside working hours)

### Summary:
- **London:** Inside working hours.
- **New York:** Outside working hours.


New York moved from 05:30 EDT to 04:30 EST, while London stayed at 09:30 GMT, because UK clocks had already changed on 25 October. The model didn't need to know any of this. The tool applied the rules.

## 7. The .py version

`client.py` does sections 4 and 6 in one script, inside `async def main()` and `asyncio.run(main())`. It reads the resource and the tools once, then loops over both dates:

<div style="font-size: 0.85em">

```python
import asyncio
import json

from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from openai import AsyncOpenAI

load_dotenv()

openai_client = AsyncOpenAI()
model = "gpt-4o-mini"
url = "http://127.0.0.1:8060/mcp"


async def get_mcp_tools(session):
    tools_result = await session.list_tools()

    tools = []
    for tool in tools_result.tools:
        tools.append(
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema,
                },
            }
        )

    return tools


async def run_conversation(session, messages, tools):
    while True:
        response = await openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
        )
        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = await session.call_tool(name, args)
            print("The model called:", name, args, "->", result.content[0].text)
            messages.append(
                {"role": "tool", "tool_call_id": tool_call.id, "content": result.content[0].text}
            )


async def main():
    async with streamable_http_client(url) as (read, write, get_session_id):
        async with ClientSession(read, write) as session:
            await session.initialize()

            offices = await session.read_resource("zones://offices")
            tools = await get_mcp_tools(session)

            for date in ["2026-10-28", "2026-11-05"]:
                prompt = await session.get_prompt("plan_meeting", {"date": date, "time": "15:00"})
                messages = [
                    {"role": "system", "content": "Our offices and their time zones:\n" + offices.contents[0].text},
                    {"role": "user", "content": prompt.messages[0].content.text},
                ]

                print(date)
                answer = await run_conversation(session, messages, tools)
                print(answer)
                print()


asyncio.run(main())
```

</div>

With the server running, `python client.py` printed:

<div style="font-size: 0.85em">

```text
2026-10-28
The model called: convert_time {'time': '2026-10-28 15:00', 'from_zone': 'Asia/Kolkata', 'to_zone': 'Europe/London'} -> 2026-10-28 09:30 GMT
The model called: convert_time {'time': '2026-10-28 15:00', 'from_zone': 'Asia/Kolkata', 'to_zone': 'America/New_York'} -> 2026-10-28 05:30 EDT
The meeting scheduled for 2026-10-28 at 15:00 Pune time converts to the following local times:

- **London**: 09:30 GMT
- **New York**: 05:30 EDT

Now, checking whether these times fall within working hours (09:00 to 18:00 local time):

- **London**: 09:30 is within working hours.
- **New York**: 05:30 is outside working hours.

In summary:
- The meeting is during working hours in London.
- The meeting is outside working hours in New York.

2026-11-05
The model called: convert_time {'time': '2026-11-05 15:00', 'from_zone': 'Asia/Kolkata', 'to_zone': 'Europe/London'} -> 2026-11-05 09:30 GMT
The model called: convert_time {'time': '2026-11-05 15:00', 'from_zone': 'Asia/Kolkata', 'to_zone': 'America/New_York'} -> 2026-11-05 04:30 EST
The meeting scheduled for **2026-11-05 at 15:00 Pune time** translates to:

- **London time:** 2026-11-05 at 09:30 (GMT) - This falls within working hours (09:00 to 18:00).
- **New York time:** 2026-11-05 at 04:30 (EST) - This is outside of working hours.

To summarize:
- **London:** Working hours (09:00 - 18:00)
- **New York:** Outside working hours
```

</div>

## Say it out loud

> "My client is the host that connects my MCP server to an OpenAI model over Streamable HTTP. It uses each primitive the way the protocol intends. It reads the `zones://offices` resource and puts it in the system message, because resources are for the app to use. It fetches the `plan_meeting` prompt and sends it as the user's message, because prompts are for the user to pick. And it converts the server's tools into OpenAI's function format, so the model decides when to call `convert_time`. Then it loops: Whenever the model asks for tools, the client runs them through the MCP session and sends the results back, until the model answers in plain text. In my run, the model called `convert_time` twice in one turn, and the answers showed New York at 05:30 EDT on 28 October and 04:30 EST on 5 November."